In [1]:
import numpy as np
from scipy.stats import norm, multivariate_normal, uniform, randint


In [2]:
# Data generation

import numpy as np
import pandas as pd

def sigmoid(x):
    """Logistic sigmoid function."""
    return 1 / (1 + np.exp(-x))

# 1. 参数设置 (Parameter Setting)
para_set = {
    'mu_Y0': -0.35, 'sigma_Y0': 0.5,
    'mu_U0': 0.35, 'sigma_U0': 0.5,
    
    # A1 的系数: intercept, Y0, U0
    'alpha_A1': np.array([-0.5, -0.35, 0.35]),
    
    # Z1 的系数: intercept, A1, Y0, U0
    'mu_Z1': np.array([0.2, 0.5, 0.5, 0.75]), 'sigma_Z1': 0.5,
    
    # W1 的系数: intercept, Y0, U0
    'mu_W1': np.array([0.2, 0.5, -0.95]), 'sigma_W1': 0.5,
    
    # Y1 的系数: intercept, A1, Y0, U0
    'mu_Y1': np.array([0.2, 0.7, 0.7, 0]), 'sigma_Y1': 0.5,
    
    # U1 的系数: intercept, A1, Y0, U0
    'mu_U1': np.array([0.2, 0.7, 0, 0.7]), 'sigma_U1': 0.5,
    
    # A2 的系数: intercept, A1, Y0, U0, Y1, U1
    # 注意：R代码中 cbind(1, A1, Y0, U0, Y1, U1) 的顺序
    'alpha_A2': np.array([-0.5, 0.5, -0.35, 0.35, -0.35, 0.35]),
    
    # Z2 的系数: intercept, Z1, A2, Y1, U1, A1, Y0, U0
    'mu_Z2': np.array([0.2, 0, 0.5, 0.5, -0.75, 0.5, 0.5, -0.75]), 'sigma_Z2': 0.5,
    
    # W2 的系数: intercept, W1, Y1, U1, Y0, U0
    'mu_W2': np.array([0.35, 0, 0.45, -0.85, 0.45, -0.85]), 'sigma_W2': 0.5,
    
    # Y2 的系数: intercept, A2, A1, A2*A1, W2, W1, Y1, U1, Y0, U0
    'mu_Y2': np.array([-1.3, 1, 1.14, 0, 0, 0, 0.5, -0.7, 0.2, -0.7]), 'sigma_Y2': 0.5
}

# 2. 观测数据生成函数 (Observational Data Generation)
def data_gen(N, para_set):
    # 生成 Y0, U0
    Y0 = np.random.normal(para_set['mu_Y0'], para_set['sigma_Y0'], N)
    U0 = np.random.normal(para_set['mu_U0'], para_set['sigma_U0'], N)
    
    # 生成 A1 (基于 propensity score 的二项分布)
    # design matrix: [1, Y0, U0]
    design_A1 = np.column_stack((np.ones(N), Y0, U0))
    prop_score_1 = 1 / (1+np.exp(np.dot(design_A1, para_set['alpha_A1'])))
    A1_bin = np.random.binomial(1, prop_score_1, N)
    # map A1 from {0,1} to {-1,1}
    A1 = 2 * A1_bin - 1 
    
    # 生成 Z1
    # design matrix: [1, A1, Y0, U0]
    design_Z1 = np.column_stack((np.ones(N), A1, Y0, U0))
    Z1 = np.dot(design_Z1, para_set['mu_Z1']) + np.random.normal(0, para_set['sigma_Z1'], N)
    
    # 生成 W1
    # design matrix: [1, Y0, U0]
    design_W1 = np.column_stack((np.ones(N), Y0, U0))
    W1 = np.dot(design_W1, para_set['mu_W1']) + np.random.normal(0, para_set['sigma_W1'], N)
    
    # 生成 U1, Y1
    # design matrix: [1, A1, Y0, U0]
    design_Y1U1 = np.column_stack((np.ones(N), A1, Y0, U0))
    Y1 = np.dot(design_Y1U1, para_set['mu_Y1']) + np.random.normal(0, para_set['sigma_Y1'], N)
    U1 = np.dot(design_Y1U1, para_set['mu_U1']) + np.random.normal(0, para_set['sigma_U1'], N)
    
    # 生成 A2
    # design matrix: [1, A1, Y0, U0, Y1, U1]
    design_A2 = np.column_stack((np.ones(N), A1, Y0, U0, Y1, U1))
    prop_score_2 = 1 / (1+np.exp(np.dot(design_A2, para_set['alpha_A2'])))
    A2_bin = np.random.binomial(1, prop_score_2, N)
    # map A2 from {0,1} to {-1,1}
    A2 = 2 * A2_bin - 1 
    
    # 生成 Z2
    # design matrix: [1, Z1, A2, Y1, U1, A1, Y0, U0]
    design_Z2 = np.column_stack((np.ones(N), Z1, A2, Y1, U1, A1, Y0, U0))
    Z2 = np.dot(design_Z2, para_set['mu_Z2']) + np.random.normal(0, para_set['sigma_Z2'], N)
    
    # 生成 W2
    # design matrix: [1, W1, Y1, U1, Y0, U0]
    design_W2 = np.column_stack((np.ones(N), W1, Y1, U1, Y0, U0))
    W2 = np.dot(design_W2, para_set['mu_W2']) + np.random.normal(0, para_set['sigma_W2'], N)
    
    # 生成 Y2
    # design matrix: [1, A2, A1, A2*A1, W2, W1, Y1, U1, Y0, U0]
    design_Y2 = np.column_stack((np.ones(N), A2, A1, A2 * A1, W2, W1, Y1, U1, Y0, U0))
    Y2 = np.dot(design_Y2, para_set['mu_Y2']) + np.random.normal(0, para_set['sigma_Y2'], N)
    
    # 组合成 DataFrame
    df = pd.DataFrame({
        'Y0': Y0, 'U0': U0, 'A1': A1, 'Z1': Z1, 'W1': W1,
        'Y1': Y1, 'U1': U1, 'A2': A2, 'Z2': Z2, 'W2': W2, 'Y2': Y2
    })
    return df

# 3. 干预（反事实）数据生成函数 (Intervened Data Generation)
def intervened_data_gen(N, para_set, a=[1, 1]):
    """
    生成在给定治疗序列 a (例如 [1, 1]) 下的反事实数据。
    这里的 A1 和 A2 不再是随机生成的，而是被强制设定为给定值。
    """
    # 生成 Y0, U0
    Y0 = np.random.normal(para_set['mu_Y0'], para_set['sigma_Y0'], N)
    U0 = np.random.normal(para_set['mu_U0'], para_set['sigma_U0'], N)
    
    # 强制设定 A1
    A1 = np.full(N, a[0])
    
    # 生成 Z1 (A1 固定)
    design_Z1 = np.column_stack((np.ones(N), A1, Y0, U0))
    Z1 = np.dot(design_Z1, para_set['mu_Z1']) + np.random.normal(0, para_set['sigma_Z1'], N)
    
    # 生成 W1
    design_W1 = np.column_stack((np.ones(N), Y0, U0))
    W1 = np.dot(design_W1, para_set['mu_W1']) + np.random.normal(0, para_set['sigma_W1'], N)
    
    # 生成 U1, Y1 (受固定的 A1 影响)
    design_Y1U1 = np.column_stack((np.ones(N), A1, Y0, U0))
    Y1 = np.dot(design_Y1U1, para_set['mu_Y1']) + np.random.normal(0, para_set['sigma_Y1'], N)
    U1 = np.dot(design_Y1U1, para_set['mu_U1']) + np.random.normal(0, para_set['sigma_U1'], N)
    
    # 强制设定 A2
    A2 = np.full(N, a[1])
    
    # 生成 Z2 (A2 固定)
    design_Z2 = np.column_stack((np.ones(N), Z1, A2, Y1, U1, A1, Y0, U0))
    Z2 = np.dot(design_Z2, para_set['mu_Z2']) + np.random.normal(0, para_set['sigma_Z2'], N)
    
    # 生成 W2
    design_W2 = np.column_stack((np.ones(N), W1, Y1, U1, Y0, U0))
    W2 = np.dot(design_W2, para_set['mu_W2']) + np.random.normal(0, para_set['sigma_W2'], N)
    
    # 生成 Y2 (反事实结果)
    design_Y2 = np.column_stack((np.ones(N), A2, A1, A2 * A1, W2, W1, Y1, U1, Y0, U0))
    Y2 = np.dot(design_Y2, para_set['mu_Y2']) + np.random.normal(0, para_set['sigma_Y2'], N)
    
    df = pd.DataFrame({
        'Y0': Y0, 'U0': U0, 'A1': A1, 'Z1': Z1, 'W1': W1,
        'Y1': Y1, 'U1': U1, 'A2': A2, 'Z2': Z2, 'W2': W2, 'Y2': Y2
    })
    return df



In [3]:
# 使用示例
if __name__ == "__main__":
    # 生成 1000 条观测数据
    df_obs = data_gen(10000, para_set)
    print("观测数据前5行:")
    print(df_obs.head())

    # 生成 1000 条反事实数据 (假设所有人都在 t=0 和 t=1 接受治疗)
    df_intervened = intervened_data_gen(1000, para_set, a=[1, 1])
    print("\n反事实数据 (a=[1,1]) 前5行:")
    print(df_intervened.head())

观测数据前5行:
         Y0        U0  A1        Z1        W1        Y1        U1  A2  \
0 -0.393558  1.044178   1  0.933083 -1.003548  1.025466  0.756205   1   
1 -0.605130  0.699271  -1  0.473982 -0.458453 -0.993347  0.083032  -1   
2 -0.441985  0.049974   1  0.328442  0.568162  0.109458  0.793418  -1   
3 -0.439905  0.894317  -1  0.402390 -1.241557 -0.358269  0.317632   1   
4  0.387785  0.436422   1  0.545525  0.481904  1.247721  0.952311  -1   

         Z2        W2        Y2  
0 -0.164743 -1.406931  0.039230  
1 -2.505927 -1.155601 -4.489035  
2 -0.165565 -0.719922 -1.254128  
3 -1.024739 -0.148559 -2.511613  
4 -0.086090 -0.088216 -1.280710  

反事实数据 (a=[1,1]) 前5行:
         Y0        U0  A1        Z1        W1        Y1        U1  A2  \
0 -1.242705  0.926431   1 -0.350481 -1.411492  0.356086  1.391260   1   
1 -0.870146  1.057528   1  0.578951 -1.521128  1.586350  1.226980   1   
2 -0.192715  0.425933   1  1.009151 -0.338780  0.223768  1.356121   1   
3 -0.430733  0.883696   1  0.27848

In [4]:
import numpy as np

def adjust_para_set_for_new_coding(original_para):
    """
    将 para_set 中的参数进行调整，以适应 A 从 {0,1} 变为 {-1,1} 的编码，
    同时保持下游变量的统计特性不变。
    """
    # 深拷贝以避免修改原字典
    new_para = original_para.copy()
    
    # 辅助函数：更新单变量依赖的系数数组
    # arr: 系数数组
    # idx_A: A 在设计矩阵中的索引位置（从0开始）
    def update_coeffs(arr, idx_A):
        new_arr = arr.copy()
        beta_A = arr[idx_A]
        # 更新截距 (总是位于索引0)
        new_arr[0] = arr[0] + 0.5 * beta_A
        # 更新 A 的系数
        new_arr[idx_A] = 0.5 * beta_A
        return new_arr

    # 1. mu_Z0: design matrix [1, A0, X0, U0]
    # A0 index = 1
    new_para['mu_Z0'] = update_coeffs(original_para['mu_Z0'], 1)
    
    # 2. mu_X1: design matrix [1, A0, X0, U0]
    # A0 index = 1
    new_para['mu_X1'] = update_coeffs(original_para['mu_X1'], 1)
    
    # 3. mu_U1: design matrix [1, A0, X0, U0]
    # A0 index = 1
    new_para['mu_U1'] = update_coeffs(original_para['mu_U1'], 1)
    
    # 4. alpha_A1: design matrix [1, A0, X0, U0, X1, U1]
    # A0 index = 1
    # 注意：这里调整是为了让 A1=1 的倾向性评分(概率)保持不变
    new_para['alpha_A1'] = update_coeffs(original_para['alpha_A1'], 1)
    
    # 5. mu_Z1: design matrix [1, Z0, A1, X1, U1, A0, X0, U0]
    # 涉及 A1 (index 2) 和 A0 (index 5)
    mu_Z1 = original_para['mu_Z1'].copy()
    beta_A1 = mu_Z1[2]
    beta_A0 = mu_Z1[5]
    
    # 截距更新：加上 A1 和 A0 的贡献
    mu_Z1[0] = mu_Z1[0] + 0.5 * beta_A1 + 0.5 * beta_A0
    # 系数更新
    mu_Z1[2] = 0.5 * beta_A1
    mu_Z1[5] = 0.5 * beta_A0
    new_para['mu_Z1'] = mu_Z1
    
    # 6. mu_Y: design matrix [1, A1, A0, A1*A0, W1, W0, X1, U1, X0, U0]
    # Indices: Int=0, A1=1, A0=2, Interaction=3
    mu_Y = original_para['mu_Y'].copy()
    beta_A1 = mu_Y[1]
    beta_A0 = mu_Y[2]
    beta_Int = mu_Y[3] # 原设定中为0
    
    # 公式转换
    # New Intercept
    mu_Y[0] = mu_Y[0] + 0.5*beta_A1 + 0.5*beta_A0 + 0.25*beta_Int
    # New A1
    mu_Y[1] = 0.5*beta_A1 + 0.25*beta_Int
    # New A0
    mu_Y[2] = 0.5*beta_A0 + 0.25*beta_Int
    # New Interaction
    mu_Y[3] = 0.25*beta_Int
    
    new_para['mu_Y'] = mu_Y
    
    return new_para

# =================使用示例=================
# 1. 定义原始参数 (原代码中的 para_set)
original_para_set = {
    'mu_X0': -0.35, 'sigma_X0': 0.5,
    'mu_U0': 0.35, 'sigma_U0': 0.5,
    'alpha_A0': np.array([-0.5, -0.35, 0.35]),
    'mu_Z0': np.array([0.2, 0.5, 0.5, 0.75]), 'sigma_Z0': 0.5,
    'mu_W0': np.array([0.2, 0.5, -0.95]), 'sigma_W0': 0.5,
    'mu_X1': np.array([0.2, 0.7, 0.7, 0]), 'sigma_X1': 0.5,
    'mu_U1': np.array([0.2, 0.7, 0, 0.7]), 'sigma_U1': 0.5,
    'alpha_A1': np.array([-0.5, 0.5, -0.35, 0.35, -0.35, 0.35]),
    'mu_Z1': np.array([0.2, 0, 0.5, 0.5, -0.75, 0.5, 0.5, -0.75]), 'sigma_Z1': 0.5,
    'mu_W1': np.array([0.35, 0, 0.45, -0.85, 0.45, -0.85]), 'sigma_W1': 0.5,
    'mu_Y': np.array([-1.3, 1, 1.14, 0, 0, 0, 0.5, -0.7, 0.2, -0.7]), 'sigma_Y': 0.5
}

# 2. 获取调整后的参数
adjusted_para_set = adjust_para_set_for_new_coding(original_para_set)

# 3. 打印对比（以 Z0 为例）
print("原 Z0 参数:", original_para_set['mu_Y'])
print("新 Z0 参数:", adjusted_para_set['mu_Y'])
# 预期结果：
# 原: [0.2, 0.5, ...] -> 0.2 + 0.5*A_old
# 新: [0.45, 0.25, ...] -> 0.45 + 0.25*A_new 
# 验证: 当 A_old=0 (A_new=-1): 0.2 vs 0.45 - 0.25 = 0.2 (一致)
# 验证: 当 A_old=1 (A_new=1): 0.2+0.5=0.7 vs 0.45 + 0.25 = 0.7 (一致)

原 Z0 参数: [-1.3   1.    1.14  0.    0.    0.    0.5  -0.7   0.2  -0.7 ]
新 Z0 参数: [-0.23  0.5   0.57  0.    0.    0.    0.5  -0.7   0.2  -0.7 ]


In [5]:
import numpy as np

# 创建一维数组
arr = np.array([1, 2, 3, 4, 5, 6])  # 形状: (6,)
reshaped = arr.reshape((3,2))  # 形状: (2, 3)

# 输出:
# [[1 2 3]
#  [4 5 6]]

# 使用 -1 自动推断
reshaped2 = reshaped.reshape(-1, 1)  # 形状: (6, 1)
print(reshaped2)

[[1]
 [2]
 [3]
 [4]
 [5]
 [6]]


In [3]:
z = np.array([1,3,-5,5])
z1 = np.sign(z)
z1

array([ 1,  1, -1,  1])

In [1]:
import torch
a = -torch.ones(20)
type(a)

torch.Tensor